In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("100_Unique_QA_Dataset.csv")

In [3]:
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [4]:
# Step 1: Tokenization --> Convert words to tokens
def tokenize(text):
  text = text.lower()
  text = text.replace('?', '') # replace special character ? with nothing
  text = text.replace("'", '')
  return text.split()

In [13]:
tokenize(df['question'][0])

['what', 'is', 'the', 'capital', 'of', 'france']

In [14]:
# Step 2: Create Vocabulary
vocab = {'<UNK>': 0}

In [26]:
def build_vocab(row):
  tokenized_question = tokenize(row['question'])
  tokenized_answer = tokenize(row['answer'])
  merged_tokens = tokenized_question + tokenized_answer
  #print(merged_tokens)
  for token in merged_tokens:
    if token not in vocab:
      vocab[token] = len(vocab)

In [27]:
df.apply(build_vocab, axis = 1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [28]:
vocab

{'<UNK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'germany': 8,
 'berlin': 9,
 'who': 10,
 'wrote': 11,
 'to': 12,
 'kill': 13,
 'a': 14,
 'mockingbird': 15,
 'harper-lee': 16,
 'largest': 17,
 'planet': 18,
 'in': 19,
 'our': 20,
 'solar': 21,
 'system': 22,
 'jupiter': 23,
 'boiling': 24,
 'point': 25,
 'water': 26,
 'celsius': 27,
 '100': 28,
 'painted': 29,
 'mona': 30,
 'lisa': 31,
 'leonardo-da-vinci': 32,
 'square': 33,
 'root': 34,
 '64': 35,
 '8': 36,
 'chemical': 37,
 'symbol': 38,
 'for': 39,
 'gold': 40,
 'au': 41,
 'which': 42,
 'year': 43,
 'did': 44,
 'world': 45,
 'war': 46,
 'ii': 47,
 'end': 48,
 '1945': 49,
 'longest': 50,
 'river': 51,
 'nile': 52,
 'japan': 53,
 'tokyo': 54,
 'developed': 55,
 'theory': 56,
 'relativity': 57,
 'albert-einstein': 58,
 'freezing': 59,
 'fahrenheit': 60,
 '32': 61,
 'known': 62,
 'as': 63,
 'red': 64,
 'mars': 65,
 'author': 66,
 '1984': 67,
 'george-orwell': 68,
 'currency': 69,
 'unit

In [29]:
len(vocab)

324

In [35]:
# Convert word to Numerical Indices
def text_to_indices(text, vocab):
  indexed_text = []
  for token in tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])
  return indexed_text

In [36]:
text_to_indices("What is Anirban.", vocab)

[1, 2, 0]

In [38]:
import torch
from torch.utils.data import Dataset, DataLoader

In [42]:
class QADataset(Dataset):
  def __init__(self, df, vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self, index):
    numerical_question = text_to_indices(self.df.iloc[index]['question'], self.vocab)
    numerical_answer = text_to_indices(self.df.iloc[index]['answer'], self.vocab)

    return torch.tensor(numerical_question), torch.tensor(numerical_answer)

In [43]:
dataset = QADataset(df, vocab)

In [44]:
dataset[10] # 10 row data ques, ans

(tensor([ 1,  2,  3,  4,  5, 53]), tensor([54]))

In [45]:
dataloader = DataLoader(dataset, batch_size = 1, shuffle = True) # since batch_size = 1 so no need to pad

In [46]:
for question, answer in dataloader:
  print(question, answer)

tensor([[ 1,  2,  3, 17, 18, 19, 20, 21, 22]]) tensor([[23]])
tensor([[ 42, 200,   2,  14, 201, 202, 203, 204]]) tensor([[205]])
tensor([[10, 29,  3, 30, 31]]) tensor([[32]])
tensor([[78, 79, 80, 81, 82, 83, 84]]) tensor([[85]])
tensor([[ 1,  2,  3, 59, 25,  5, 26, 19, 60]]) tensor([[61]])
tensor([[  1,   2,   3, 122, 123,  19,   3,  45]]) tensor([[124]])
tensor([[ 42, 137,   2, 226,  12,   3, 227, 228]]) tensor([[155]])
tensor([[ 10,  75, 208]]) tensor([[209]])
tensor([[ 10,  11, 189, 158, 190]]) tensor([[191]])
tensor([[  1,   2,   3,  69,   5, 155]]) tensor([[156]])
tensor([[ 10,   2,  62,  63,   3, 283,   5, 284]]) tensor([[285]])
tensor([[ 78,  79, 195,  81,  19,   3, 196, 197, 198]]) tensor([[199]])
tensor([[ 42, 101,   2,   3,  17]]) tensor([[102]])
tensor([[ 78,  79, 261, 151,  14, 262, 153]]) tensor([[36]])
tensor([[ 78,  79, 288,  81,  19,  14, 289]]) tensor([[85]])
tensor([[ 42, 299, 300, 118,  14, 301, 302, 158, 303, 304, 305, 306]]) tensor([[307]])
tensor([[10, 96,  3, 97]

### RNN Architecture
- 1 Input Layer $\rightarrow$ 50 dimensional word embedding vector
- 1 Hidden Layer $\rightarrow$ 64 neurons
- 1 Output Layer $\rightarrow$ 324 Neurons $\rightarrow$ equal to length of vocab

In [47]:
import torch.nn as nn

In [84]:
class SimpleRNN(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim = 50)
    self.rnn = nn.RNN(50, 64, batch_first = True)
    # we cant use Sequential() because from RNN layer we are getting multiple outputs (from Hidden States Oi and Final output)
    self.fc = nn.Linear(64, vocab_size)

  def forward(self, question):
    embedded_question = self.embedding(question)
    hidden, final = self.rnn(embedded_question)
    output = self.fc(final.squeeze(0))
    return output


In [53]:
# what does embedding layer do
x = nn.Embedding(324, embedding_dim = 50)
a = x(dataset[0][0]) # 0th row question
print(a)
# We get 6 vectors (number of words = 6) each of dimension 50

tensor([[-2.7905e-01,  6.2512e-01,  1.4794e+00,  1.0353e+00,  9.1692e-02,
         -7.8963e-01, -1.8489e+00, -2.1390e+00, -4.1245e-02, -7.4883e-01,
         -9.2151e-01,  2.3164e+00,  1.4413e+00, -8.1684e-01, -7.9389e-01,
          1.0044e+00, -1.0133e+00,  7.1586e-01,  9.9728e-01,  7.6236e-01,
          1.1985e+00, -5.9986e-01, -8.4827e-01, -1.4747e+00, -1.8072e-01,
          1.8730e+00,  8.2530e-01, -1.3752e+00, -1.9957e+00, -5.3346e-01,
          6.9772e-02,  1.2872e+00,  5.9771e-01,  1.5885e+00, -2.8372e-02,
          8.4699e-02, -1.4164e+00, -6.0629e-01, -2.8715e-01, -4.1684e-01,
          9.5565e-02, -4.0339e-01, -9.6968e-01,  3.9930e-01,  3.7131e-01,
         -7.6722e-01, -4.5726e-01, -1.8362e-01,  1.8718e-02,  2.6434e+00],
        [ 8.7164e-01,  1.2764e+00, -1.0232e+00, -4.4587e-01,  7.0637e-01,
         -1.1038e-01,  2.1126e-01, -9.3152e-01,  1.2357e-01, -4.2788e-01,
          5.7965e-01,  1.1039e+00, -2.2819e+00,  1.5913e-01, -2.3811e+00,
         -3.9384e-01, -3.3723e-01, -2

In [54]:
# pass a to RNN
y = nn.RNN(50, 64)
y(a)

(tensor([[-0.7040, -0.9314,  0.0741,  0.0077,  0.7504, -0.1132,  0.1083,  0.7786,
           0.6258, -0.6420,  0.0088,  0.0765, -0.1437, -0.1037,  0.1155, -0.1899,
          -0.2021,  0.1992, -0.1137,  0.2365, -0.0367,  0.0278,  0.7363,  0.2932,
           0.0283, -0.6160, -0.2954, -0.5248, -0.0421, -0.4646,  0.4780, -0.7373,
           0.0935, -0.6667,  0.4280, -0.4415, -0.6290, -0.0988,  0.3242,  0.3705,
          -0.1052, -0.0518, -0.2714,  0.1657, -0.3450,  0.1493,  0.6060,  0.5957,
          -0.1814,  0.5313, -0.8127, -0.2346, -0.6982, -0.0517,  0.6013,  0.3790,
           0.7605, -0.5846, -0.4390,  0.6628,  0.8923,  0.7848, -0.6218,  0.8698],
         [-0.0201,  0.0301, -0.1349, -0.0869, -0.4851,  0.4822, -0.0801,  0.6735,
           0.5846,  0.4130,  0.0567,  0.4999, -0.6614, -0.1685, -0.2273,  0.2973,
           0.7094, -0.1304,  0.7451,  0.0194, -0.4652,  0.1721,  0.6252, -0.4003,
           0.9004, -0.7517, -0.2790, -0.2611,  0.1036, -0.6107, -0.6646, -0.2359,
           0.19

In [59]:
y(a)[0].shape # Outputs o1, o2, o3, o4, o5, o6 Hidden States one after each time step

torch.Size([6, 64])

In [61]:
b = y(a)[1]
y(a)[1].shape # Final output from RNN

torch.Size([1, 64])

In [63]:
# send final output of RNN to FC Layer
z = nn.Linear(64, len(vocab))
z(b)

tensor([[-1.4381e-01, -3.2791e-01,  1.8895e-01, -1.4841e-01,  1.7153e-01,
         -4.3273e-01,  1.3513e-01, -2.2851e-02,  2.7803e-01,  2.2004e-01,
          4.9429e-02,  6.8505e-02, -1.6752e-01, -2.8888e-01, -1.3402e-01,
          5.5633e-01,  3.6930e-02, -6.2164e-02, -3.8009e-01,  5.2007e-01,
         -1.5454e-01, -3.3636e-01, -5.4472e-01,  8.5189e-01,  3.5697e-01,
         -4.3598e-01,  5.4630e-02,  5.5763e-01, -1.7627e-01, -7.4644e-02,
          1.3781e-01, -4.1716e-01, -4.5293e-02, -7.4606e-02, -3.7268e-01,
         -1.9378e-01, -1.9878e-01,  9.2432e-02, -6.9958e-01,  2.1003e-01,
         -1.2528e-01, -1.4050e-01, -2.7490e-01,  9.3842e-04, -5.6583e-01,
          4.1692e-01, -1.3176e-01,  1.9863e-01, -1.8566e-01, -1.6762e-01,
          2.4905e-01, -1.2081e-01, -1.4267e-01, -1.2773e-01, -2.4096e-02,
         -1.9329e-01,  2.6130e-01, -3.8974e-01, -1.9920e-02, -9.6180e-02,
          6.1497e-01,  1.7024e-01, -1.2480e-01,  2.2202e-04,  2.9790e-01,
          2.7844e-01, -5.3512e-01, -2.

In [64]:
z(b).shape

torch.Size([1, 324])

## Training Process

In [72]:
####Debugging
x = nn.Embedding(324, embedding_dim=50)
y = nn.RNN(50, 64, batch_first=True)
z = nn.Linear(64, 324)

a = dataset[0][0].reshape(1,6)
print("shape of a:", a.shape)
b = x(a)
print("shape of b:", b.shape)
c, d = y(b)
print("shape of c:", c.shape)
print("shape of d:", d.shape)

e = z(d.squeeze(0))

print("shape of e:", e.shape)

shape of a: torch.Size([1, 6])
shape of b: torch.Size([1, 6, 50])
shape of c: torch.Size([1, 6, 64])
shape of d: torch.Size([1, 1, 64])
shape of e: torch.Size([1, 324])


In [85]:
learning_rate = 0.01
epochs = 20

In [86]:
model = SimpleRNN(len(vocab))

In [87]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)

In [88]:
# training loop
for epoch in range(epochs):
  total_loss = 0
  for question, answer in dataloader:
    optimizer.zero_grad()

    # forward pass
    output = model(question)

    # calculate loss
    loss = criterion(output, answer[0])

    # back prop
    loss.backward()

    # update
    optimizer.step()

    total_loss += loss.item()

  print(f"Epochs: {epoch}, Loss: {total_loss:4f}")

Epochs: 0, Loss: 540.139019
Epochs: 1, Loss: 339.298120
Epochs: 2, Loss: 144.944890
Epochs: 3, Loss: 61.078041
Epochs: 4, Loss: 38.320868
Epochs: 5, Loss: 32.828745
Epochs: 6, Loss: 26.657410
Epochs: 7, Loss: 21.890049
Epochs: 8, Loss: 13.538467
Epochs: 9, Loss: 12.841703
Epochs: 10, Loss: 11.667822
Epochs: 11, Loss: 7.625669
Epochs: 12, Loss: 5.223808
Epochs: 13, Loss: 6.598932
Epochs: 14, Loss: 10.092472
Epochs: 15, Loss: 5.038844
Epochs: 16, Loss: 10.149715
Epochs: 17, Loss: 3.942872
Epochs: 18, Loss: 0.764561
Epochs: 19, Loss: 0.436354


In [96]:
def predict(model, question, threshold=0.5):

  # convert question to numbers
  numerical_question = text_to_indices(question, vocab)

  # tensor
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)

  # send to model
  output = model(question_tensor)

  # convert logits to probs
  probs = torch.nn.functional.softmax(output, dim=1)

  # find index of max prob
  value, index = torch.max(probs, dim=1)

  if value < threshold:
    print("I don't know")
    return

  print(value, index)
  print(list(vocab.keys())[index])

In [97]:
predict(model, "What is the largest planet in our solar system?")

tensor([0.9922], grad_fn=<MaxBackward0>) tensor([23])
jupiter


In [98]:
predict(model, "Who is Anirban?")

I don't know
